# 🚀 ECG MULTI-LABEL CLASSIFICATION - ENHANCED TRAINING

## Tích hợp 4 giải pháp nâng cao:
1. ✅ **Post-processing Rules** cho AFIB/AF
2. ✅ **LVH-specific Augmentation** (giữ nguyên biên độ)
3. ✅ **Focal Loss** cho class imbalance
4. ✅ **Adaptive Oversampling** với balanced sampler

**Kỳ vọng cải thiện:**
- LVH F1: 0.47 → **0.70-0.78** (+50%)
- AFIB F1: 0.33 → **0.55-0.63** (+67%)
- AF F1: 0.66 → **0.75-0.80** (+14%)
- Micro F1: 0.87 → **0.91-0.93** (+5%)

---
## 📦 STEP 0: Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    multilabel_confusion_matrix,
    f1_score,
    precision_recall_fscore_support
)
from scipy import signal as scipy_signal
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

🖥️ Using device: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   Memory: 4.00 GB


---
## 🔧 STEP 1: Enhanced Augmentation Functions

### ⚠️ QUAN TRỌNG: LVH cần augmentation đặc biệt!

In [2]:
# ============= AUGMENTATION FUNCTIONS =============

def lvh_preserving_augmentation(signal):
    """
    🎯 AUGMENTATION ĐẶC BIỆT CHO LVH
    
    LVH được chẩn đoán dựa trên BIÊN ĐỘ TUYỆT ĐỐI của sóng R/S.
    → KHÔNG ĐƯỢC dùng scaling (sẽ phá hủy thông tin biên độ)
    → Chỉ thay đổi: timing, baseline wander, noise nhẹ
    
    Args:
        signal: (12, 5000) hoặc (5000, 12)
    """
    # Detect shape and adjust
    if signal.shape[0] == 12:  # (12, 5000)
        axis = 1
    else:  # (5000, 12)
        axis = 0
    
    # 1. Baseline wander (drift chậm)
    baseline_length = signal.shape[axis]
    baseline = np.random.randn(baseline_length) * 0.003
    baseline = np.cumsum(baseline) * 0.0003
    
    if axis == 1:
        signal = signal + baseline[np.newaxis, :]
    else:
        signal = signal + baseline[:, np.newaxis]
    
    # 2. Time shift nhẹ (không ảnh hưởng biên độ)
    shift = np.random.randint(-30, 30)
    signal = np.roll(signal, shift, axis=axis)
    
    # 3. Gaussian noise nhẹ (rất nhẹ để không che mất sóng)
    noise = np.random.normal(0, 0.003, signal.shape)
    signal = signal + noise
    
    # 4. ❌ KHÔNG DÙNG scaling, rotation, hoặc bất kỳ phép biến đổi nào ảnh hưởng biên độ!
    
    return signal


def rhythm_augmentation(signal, strength="normal"):
    """
    🎵 AUGMENTATION CHO CÁC BỆNH VỀ NHỊP (AFIB, AF, ST, SB, SVT, SR)
    
    Các bệnh về nhịp KHÔNG phụ thuộc vào biên độ tuyệt đối
    → Có thể dùng scaling, jittering mạnh hơn
    
    Args:
        signal: (12, 5000) hoặc (5000, 12)
        strength: "normal" hoặc "aggressive"
    """
    # Detect shape
    if signal.shape[0] == 12:
        axis = 1
    else:
        axis = 0
    
    if strength == "aggressive":
        # Aggressive augmentation for rare classes (AFIB, SVT)
        num_augments = np.random.randint(2, 4)
        noise_sigma = np.random.uniform(0.01, 0.03)
        scale_sigma = np.random.uniform(0.1, 0.2)
        max_shift = np.random.uniform(0.05, 0.1)
    else:
        # Normal augmentation
        num_augments = 1
        noise_sigma = 0.01
        scale_sigma = 0.1
        max_shift = 0.05
    
    for _ in range(num_augments):
        # 1. Jittering (Gaussian noise)
        noise = np.random.normal(0, noise_sigma, signal.shape)
        signal = signal + noise
        
        # 2. Scaling (OK for rhythm diseases)
        scale = np.random.normal(1.0, scale_sigma)
        scale = np.clip(scale, 0.7, 1.3)  # Giới hạn để không quá cực đoan
        signal = signal * scale
        
        # 3. Time shift
        seq_len = signal.shape[axis]
        shift = np.random.randint(-int(max_shift * seq_len), int(max_shift * seq_len))
        signal = np.roll(signal, shift, axis=axis)
        
        # 4. Baseline wander
        baseline_length = signal.shape[axis]
        baseline = np.random.randn(baseline_length) * 0.01
        baseline = np.cumsum(baseline) * 0.001
        
        if axis == 1:
            signal = signal + baseline[np.newaxis, :]
        else:
            signal = signal + baseline[:, np.newaxis]
        
        # 5. Random dropout (simulate missing data) - chỉ cho aggressive
        if strength == "aggressive" and np.random.rand() > 0.5:
            dropout_rate = 0.03  # 3% dropout
            dropout_mask = np.random.rand(*signal.shape) > dropout_rate
            signal = signal * dropout_mask
    
    # Clip to reasonable range
    signal = np.clip(signal, -3, 3)
    
    return signal


# Test augmentation functions
print("✅ Augmentation functions loaded")
print("   - lvh_preserving_augmentation(): Giữ nguyên biên độ cho LVH")
print("   - rhythm_augmentation(): Augmentation mạnh cho AFIB, AF, ST, SB, SVT")

✅ Augmentation functions loaded
   - lvh_preserving_augmentation(): Giữ nguyên biên độ cho LVH
   - rhythm_augmentation(): Augmentation mạnh cho AFIB, AF, ST, SB, SVT


---
## 🎯 STEP 2: Enhanced Dataset Class

In [3]:
# ============= ENHANCED DATASET =============

class EnhancedECGDataset(Dataset):
    """
    Dataset với class-specific augmentation:
    - LVH: lvh_preserving_augmentation (không phá hủy biên độ)
    - AFIB, AF, SVT: rhythm_augmentation với strength="aggressive"
    - SB, ST, SR: rhythm_augmentation với strength="normal"
    """
    def __init__(self, file_paths, labels, augment=False, 
                 label_names=None, lvh_index=6):
        """
        Args:
            file_paths: Array of .npy file paths
            labels: (n_samples, n_classes) binary labels
            augment: Whether to apply augmentation
            label_names: List of label names
            lvh_index: Index của LVH trong labels (mặc định 6)
        """
        self.file_paths = file_paths
        self.labels = labels
        self.augment = augment
        self.lvh_index = lvh_index
        
        # Identify rare classes for aggressive augmentation
        class_counts = labels.sum(axis=0)
        total_samples = len(labels)
        
        # Classes < 5% of dataset are considered rare
        rare_threshold = total_samples * 0.05
        self.rare_class_indices = np.where(class_counts < rare_threshold)[0]
        
        # Pre-compute which samples contain rare classes (for speed)
        if len(self.rare_class_indices) > 0:
            rare_mask = labels[:, self.rare_class_indices].sum(axis=1) > 0
            self.is_rare = rare_mask
        else:
            self.is_rare = np.zeros(len(labels), dtype=bool)
        
        # Pre-compute which samples are LVH
        self.is_lvh = labels[:, lvh_index] == 1
        
        print(f"📊 Dataset initialized:")
        print(f"   Total samples: {len(file_paths)}")
        print(f"   Rare classes (< 5%): {self.rare_class_indices}")
        print(f"   LVH samples: {self.is_lvh.sum()}")
        print(f"   Rare samples (total): {self.is_rare.sum()}")
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        # Load signal
        signal = np.load(self.file_paths[idx]).astype(np.float32)
        signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Apply augmentation if enabled
        if self.augment:
            # Strategy 1: LVH → Preserve amplitude
            if self.is_lvh[idx]:
                signal = lvh_preserving_augmentation(signal)
            
            # Strategy 2: Rare rhythm classes → Aggressive augmentation
            elif self.is_rare[idx]:
                signal = rhythm_augmentation(signal, strength="aggressive")
            
            # Strategy 3: Common classes → Normal augmentation
            else:
                signal = rhythm_augmentation(signal, strength="normal")
        
        # Convert to tensor
        signal = torch.tensor(signal, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return signal, label

print("✅ EnhancedECGDataset class loaded")

✅ EnhancedECGDataset class loaded


---
## ⚖️ STEP 3: Balanced Sampler với Adaptive Oversampling

In [4]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Sampler

print("🔥 Loading AGGRESSIVE versions of Sampler, Loss, and Weights...")

# ============= CELL 3: AGGRESSIVE BALANCED SAMPLER =============

class AggressiveBalancedSampler(Sampler):
    """
    🚀 AGGRESSIVE oversampling for EXTREME class imbalance.
    
    Strategy:
    - Very rare classes (< 2%): Make them ~15% of dataset
    - Rare classes (2-5%): Make them ~10% of dataset
    - Common classes: Keep as is
    
    Example:
        LVH (528 samples, 1.5%) → ~5,300 samples (15%) = 10x oversample
        AFIB (1,456 samples, 4.1%) → ~3,500 samples (10%) = 2.4x oversample
    """
    def __init__(self, labels, target_very_rare_ratio=0.15, target_rare_ratio=0.10):
        """
        Args:
            labels: (n_samples, n_classes) binary array
            target_very_rare_ratio: Target ratio for very rare classes (default 15%)
            target_rare_ratio: Target ratio for rare classes (default 10%)
        """
        self.labels = labels
        self.num_samples = len(labels)
        self.num_classes = labels.shape[1]
        
        # Class frequencies
        class_counts = labels.sum(axis=0)
        
        # Define thresholds
        very_rare_threshold = self.num_samples * 0.02  # < 2%
        rare_threshold = self.num_samples * 0.05       # < 5%
        
        # Identify classes
        very_rare_classes = np.where(class_counts < very_rare_threshold)[0]
        rare_classes = np.where(
            (class_counts >= very_rare_threshold) & 
            (class_counts < rare_threshold)
        )[0]
        
        # Find samples
        very_rare_mask = labels[:, very_rare_classes].sum(axis=1) > 0
        rare_mask = labels[:, rare_classes].sum(axis=1) > 0
        
        very_rare_indices = np.where(very_rare_mask)[0]
        rare_indices = np.where(rare_mask & ~very_rare_mask)[0]
        
        # Calculate target counts
        target_very_rare_count = int(self.num_samples * target_very_rare_ratio)
        target_rare_count = int(self.num_samples * target_rare_ratio)
        
        # Calculate oversampling needed
        n_oversample_very_rare = max(0, target_very_rare_count - len(very_rare_indices))
        n_oversample_rare = max(0, target_rare_count - len(rare_indices))
        
        # Create final indices
        self.indices = np.concatenate([
            np.arange(self.num_samples),  # All original samples
            np.random.choice(very_rare_indices, size=n_oversample_very_rare, replace=True),
            np.random.choice(rare_indices, size=n_oversample_rare, replace=True)
        ])
        
        np.random.shuffle(self.indices)
        
        # Calculate statistics
        total_after = len(self.indices)
        oversample_ratio = total_after / self.num_samples
        
        actual_very_rare = sum(1 for idx in self.indices 
                              if labels[idx, very_rare_classes].sum() > 0)
        actual_rare = sum(1 for idx in self.indices 
                         if labels[idx, rare_classes].sum() > 0)
        
        print(f"\n{'='*70}")
        print(f"📊 AGGRESSIVE Balanced Sampler Statistics")
        print(f"{'='*70}")
        print(f"   Original samples: {self.num_samples:,}")
        print(f"   After oversampling: {total_after:,}")
        print(f"   Overall ratio: {oversample_ratio:.2f}x")
        
        print(f"\n   🔴 Very rare classes (< 2%): {very_rare_classes}")
        for cls in very_rare_classes:
            orig_count = class_counts[cls]
            after_count = sum(1 for idx in self.indices if labels[idx, cls] == 1)
            factor = after_count / orig_count if orig_count > 0 else 0
            pct_after = after_count / total_after * 100
            
            print(f"      Class {cls}:")
            print(f"         Before: {orig_count:.0f} samples ({orig_count/self.num_samples*100:.2f}%)")
            print(f"         After:  {after_count:.0f} samples ({pct_after:.2f}%)")
            print(f"         Factor: {factor:.1f}x")
        
        print(f"\n   🟡 Rare classes (2-5%): {rare_classes}")
        for cls in rare_classes:
            orig_count = class_counts[cls]
            after_count = sum(1 for idx in self.indices if labels[idx, cls] == 1)
            factor = after_count / orig_count if orig_count > 0 else 0
            pct_after = after_count / total_after * 100
            
            print(f"      Class {cls}:")
            print(f"         Before: {orig_count:.0f} samples ({orig_count/self.num_samples*100:.2f}%)")
            print(f"         After:  {after_count:.0f} samples ({pct_after:.2f}%)")
            print(f"         Factor: {factor:.1f}x")
        
        print(f"\n   ✅ Very rare samples in dataset: {actual_very_rare:,} ({actual_very_rare/total_after*100:.1f}%)")
        print(f"   ✅ Rare samples in dataset: {actual_rare:,} ({actual_rare/total_after*100:.1f}%)")
        print(f"{'='*70}\n")
    
    def __iter__(self):
        return iter(self.indices)
    
    def __len__(self):
        return len(self.indices)

print("✅ AggressiveBalancedSampler loaded")

🔥 Loading AGGRESSIVE versions of Sampler, Loss, and Weights...
✅ AggressiveBalancedSampler loaded


---
## 🔥 STEP 4: Focal Loss Implementation

In [5]:
# ============= CELL 4: AGGRESSIVE FOCAL LOSS =============

class AggressiveFocalLoss(nn.Module):
    """
    🔥 AGGRESSIVE Focal Loss for extreme class imbalance.
    
    Changes from standard:
    - Higher gamma (2.5 instead of 2.0) → Focus MORE on hard examples
    - Supports class weights with higher range
    """
    def __init__(self, alpha=0.25, gamma=2.5, class_weights=None, reduction='mean'):
        super(AggressiveFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: (batch_size, num_classes) - logits
            targets: (batch_size, num_classes) - binary labels
        """
        # Binary cross entropy
        BCE_loss = F.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none'
        )
        
        # Focal term - higher gamma means MORE focus on hard examples
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        
        # Apply class weights
        if self.class_weights is not None:
            weights = self.class_weights.unsqueeze(0).to(inputs.device)
            focal_loss = focal_loss * weights
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("✅ AggressiveFocalLoss loaded")

✅ AggressiveFocalLoss loaded


---
## 🔍 STEP 5: AFIB/AF Separator (Post-processing)

In [6]:
# ============= CELL 5: AGGRESSIVE CLASS WEIGHTS =============

def calculate_aggressive_weights(y_train, power=2.0, min_weight=0.1, max_weight=50.0):
    """
    🔥 Calculate AGGRESSIVE class weights for extreme imbalance.
    
    Formula: weight = (total_samples / class_count) ^ power
    
    Args:
        y_train: (n_samples, n_classes) binary labels
        power: Exponent (higher = more aggressive)
               1.0 = inverse frequency (standard)
               2.0 = inverse squared (AGGRESSIVE - recommended)
               3.0 = inverse cubed (VERY AGGRESSIVE)
        min_weight: Minimum weight (prevent too low)
        max_weight: Maximum weight (prevent instability)
    
    Returns:
        weights: (n_classes,) array
    """
    class_counts = y_train.sum(axis=0)
    total_samples = len(y_train)
    
    # Calculate weights with power
    weights = (total_samples / (class_counts + 1e-6)) ** power
    
    # Normalize to sum = num_classes
    weights = weights / weights.sum() * len(weights)
    
    # Clip to safe range
    weights = np.clip(weights, min_weight, max_weight)
    
    # Print statistics
    print(f"\n{'='*70}")
    print(f"📊 AGGRESSIVE Class Weights (power={power})")
    print(f"{'='*70}")
    print(f"{'Class':<8} {'Samples':<10} {'Freq %':<10} {'Weight':<10} {'vs Avg':<10}")
    print(f"{'-'*70}")
    
    avg_weight = weights.mean()
    for i, (count, weight) in enumerate(zip(class_counts, weights)):
        freq_pct = count / total_samples * 100
        vs_avg = weight / avg_weight
        
        # Color coding
        if weight > avg_weight * 3:
            marker = "🔴"
        elif weight > avg_weight * 1.5:
            marker = "🟡"
        else:
            marker = "🟢"
        
        print(f"{marker} {i:<6} {count:<10.0f} {freq_pct:<10.2f} {weight:<10.2f} {vs_avg:<10.1f}x")
    
    print(f"{'-'*70}")
    print(f"Average weight: {avg_weight:.2f}")
    print(f"Max weight: {weights.max():.2f} (class {weights.argmax()})")
    print(f"Min weight: {weights.min():.2f} (class {weights.argmin()})")
    print(f"{'='*70}\n")
    
    return weights

print("✅ calculate_aggressive_weights loaded")

✅ calculate_aggressive_weights loaded


---
## 📚 STEP 6: Load Your Data

### ⚠️ QUAN TRỌNG: Thay đổi đường dẫn phù hợp với dự án của bạn!

In [7]:
# ============= LOAD DATA =============

# 🔧 THAY ĐỔI ĐƯỜNG DẪN NÀY:
data_dir = "E:/NCKH - 2026/ECG Project/data/processed/splits_leads12_050125_scale"  # ← SỬA ĐỔI

# Load splits (giả sử bạn đã có từ split_dataset.py)
X_train = np.load(os.path.join(data_dir, "train_files.npy"), allow_pickle=True)
X_val = np.load(os.path.join(data_dir, "val_files.npy"), allow_pickle=True)
X_test = np.load(os.path.join(data_dir, "test_files.npy"), allow_pickle=True)

y_train = np.load(os.path.join(data_dir, "y_train.npy"))
y_val = np.load(os.path.join(data_dir, "y_val.npy"))
y_test = np.load(os.path.join(data_dir, "y_test.npy"))

# Label names (điều chỉnh nếu khác)
label_names = [
    "SB",    # 0: Nhịp chậm xoang
    "SR",    # 1: Nhịp xoang bình thường
    "AF",    # 2: Cuồng nhĩ
    "AFIB",  # 3: Rung nhĩ
    "SVT",   # 4: Nhịp nhanh trên thất
    "ST",    # 5: Nhịp xoang nhanh
    "LVH"    # 6: Phì đại thất trái
]

LVH_INDEX = 6  # Index của LVH
AFIB_INDEX = 3
AF_INDEX = 2

# Print statistics
print(f"📊 Data loaded successfully:")
print(f"   Train: {len(X_train)} samples")
print(f"   Val:   {len(X_val)} samples")
print(f"   Test:  {len(X_test)} samples")
print(f"\n   Number of classes: {y_train.shape[1]}")
print(f"   Labels: {label_names}")

# Class distribution
print(f"\n📈 Class distribution (train set):")
class_counts = y_train.sum(axis=0)
for i, (name, count) in enumerate(zip(label_names, class_counts)):
    pct = count / len(y_train) * 100
    print(f"   {name:6s}: {count:5.0f} samples ({pct:5.2f}%)")

📊 Data loaded successfully:
   Train: 35361 samples
   Val:   4421 samples
   Test:  4421 samples

   Number of classes: 7
   Labels: ['SB', 'SR', 'AF', 'AFIB', 'SVT', 'ST', 'LVH']

📈 Class distribution (train set):
   SB    : 13203 samples (37.34%)
   SR    :  6382 samples (18.05%)
   AF    :  6407 samples (18.12%)
   AFIB  :  1456 samples ( 4.12%)
   SVT   :   546 samples ( 1.54%)
   ST    :  5757 samples (16.28%)
   LVH   :   528 samples ( 1.49%)


---
## 🏗️ STEP 7: Create Datasets and Dataloaders

In [8]:
# ============= CELL 7: CREATE DATALOADERS (UPDATED) =============

print("\n📦 Creating datasets and dataloaders...")

# Train dataset (keep same - already has augmentation)
print("   Creating train_dataset...")
train_dataset = EnhancedECGDataset(
    X_train, y_train, 
    augment=True,
    label_names=label_names,
    lvh_index=LVH_INDEX
)

# Val and test datasets (no augmentation)
print("   Creating val_dataset...")
val_dataset = EnhancedECGDataset(
    X_val, y_val,
    augment=False,
    label_names=label_names,
    lvh_index=LVH_INDEX
)

print("   Creating test_dataset...")
test_dataset = EnhancedECGDataset(
    X_test, y_test,
    augment=False,
    label_names=label_names,
    lvh_index=LVH_INDEX
)

print("\n🔥 Creating AGGRESSIVE balanced sampler...")
# Create aggressive sampler
balanced_sampler = AggressiveBalancedSampler(
    y_train, 
    target_very_rare_ratio=0.15,  # LVH will be ~15% of dataset
    target_rare_ratio=0.10        # AFIB will be ~10% of dataset
)

print("\n📊 Creating dataloaders...")
# Training loader with aggressive sampling
train_loader = DataLoader(
    train_dataset,
    batch_size=16,  # ← REDUCED from 32 to 16 for better gradient
    sampler=balanced_sampler,
    num_workers=0,
    pin_memory=True
)

# Validation and test loaders (no changes)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"\n✅ Dataloaders created:")
print(f"   Train batches: {len(train_loader)} (batch_size=16)")
print(f"   Val batches:   {len(val_loader)} (batch_size=64)")
print(f"   Test batches:  {len(test_loader)} (batch_size=64)")

# Verify first batch
print(f"\n🔍 Verifying first training batch...")
test_iter = iter(train_loader)
test_signals, test_labels = next(test_iter)

batch_counts = test_labels.sum(dim=0).cpu().numpy()
print(f"   Batch size: {test_signals.shape[0]}")
print(f"   Classes in first batch:")
for i, (name, count) in enumerate(zip(label_names, batch_counts)):
    marker = "🔴" if count >= 3 else ("🟡" if count >= 1 else "⚪")
    print(f"      {marker} {name:6s}: {count:.0f} samples")

print(f"\n   ⭐ AFIB in batch: {batch_counts[AFIB_INDEX]:.0f} (expected: 2-3)")
print(f"   ⭐ LVH in batch:  {batch_counts[LVH_INDEX]:.0f} (expected: 2-3)")

if batch_counts[LVH_INDEX] < 2:
    print(f"\n   ⚠️ WARNING: LVH count still low. May need higher target_very_rare_ratio")
    print(f"      Try: target_very_rare_ratio=0.20 (20% instead of 15%)")

print("\n" + "="*80)



📦 Creating datasets and dataloaders...
   Creating train_dataset...
📊 Dataset initialized:
   Total samples: 35361
   Rare classes (< 5%): [3 4 6]
   LVH samples: 528
   Rare samples (total): 2518
   Creating val_dataset...
📊 Dataset initialized:
   Total samples: 4421
   Rare classes (< 5%): [3 4 6]
   LVH samples: 58
   Rare samples (total): 296
   Creating test_dataset...
📊 Dataset initialized:
   Total samples: 4421
   Rare classes (< 5%): [3 4 6]
   LVH samples: 52
   Rare samples (total): 277

🔥 Creating AGGRESSIVE balanced sampler...

📊 AGGRESSIVE Balanced Sampler Statistics
   Original samples: 35,361
   After oversampling: 41,683
   Overall ratio: 1.18x

   🔴 Very rare classes (< 2%): [4 6]
      Class 4:
         Before: 546 samples (1.54%)
         After:  2719 samples (6.52%)
         Factor: 5.0x
      Class 6:
         Before: 528 samples (1.49%)
         After:  2599 samples (6.24%)
         Factor: 4.9x

   🟡 Rare classes (2-5%): [3]
      Class 3:
         Before: 145

---

In [11]:
# ========== DEBUG CELL - THÊM VÀO SAU CELL 7 ==========

import torch
import numpy as np

print("="*80)
print("🔍 DEBUGGING - Kiểm tra data và model")
print("="*80)

# ===== 1. Kiểm tra Label Distribution =====
print("\n1️⃣ LABEL DISTRIBUTION (Training Set)")
print("-" * 50)

class_counts = y_train.sum(axis=0)
total_samples = len(y_train)

for i, (name, count) in enumerate(zip(label_names, class_counts)):
    pct = count / total_samples * 100
    print(f"   {i}: {name:6s} = {count:5.0f} samples ({pct:5.2f}%)")

print(f"\n   AFIB index: {AFIB_INDEX}, count: {class_counts[AFIB_INDEX]}")
print(f"   LVH index: {LVH_INDEX}, count: {class_counts[LVH_INDEX]}")

# ===== 2. Kiểm tra một batch từ DataLoader =====
print("\n2️⃣ CHECKING FIRST BATCH FROM TRAIN LOADER")
print("-" * 50)

# Get first batch
train_iter = iter(train_loader)
signals_batch, labels_batch = next(train_iter)

print(f"   Batch shape: {signals_batch.shape}")
print(f"   Labels shape: {labels_batch.shape}")

# Check labels in this batch
labels_np = labels_batch.numpy()
batch_class_counts = labels_np.sum(axis=0)

print(f"\n   Classes in this batch:")
for i, (name, count) in enumerate(zip(label_names, batch_class_counts)):
    print(f"      {i}: {name:6s} = {count:.0f} samples")

print(f"\n   ✅ AFIB samples in batch: {batch_class_counts[AFIB_INDEX]:.0f}")
print(f"   ✅ LVH samples in batch: {batch_class_counts[LVH_INDEX]:.0f}")

# ===== 3. Kiểm tra Model Output =====
print("\n3️⃣ CHECKING MODEL OUTPUT")
print("-" * 50)

model.eval()
with torch.no_grad():
    signals_device = signals_batch[:8].to(device)  # Take 8 samples
    outputs = model(signals_device)
    probs = torch.sigmoid(outputs)
    
print(f"   Logits shape: {outputs.shape}")
print(f"   Logits range: [{outputs.min().item():.4f}, {outputs.max().item():.4f}]")
print(f"   Probs range: [{probs.min().item():.4f}, {probs.max().item():.4f}]")

print(f"\n   Mean probability per class (first 8 samples):")
mean_probs = probs.mean(dim=0).cpu().numpy()
for i, (name, prob) in enumerate(zip(label_names, mean_probs)):
    print(f"      {i}: {name:6s} = {prob:.4f}")

print(f"\n   ⚠️ AFIB mean prob: {mean_probs[AFIB_INDEX]:.4f}")
print(f"   ⚠️ LVH mean prob: {mean_probs[LVH_INDEX]:.4f}")

# ===== 4. Kiểm tra Class Weights =====
print("\n4️⃣ CHECKING CLASS WEIGHTS")
print("-" * 50)

if hasattr(criterion, 'class_weights') and criterion.class_weights is not None:
    weights = criterion.class_weights.cpu().numpy()
    print(f"   Class weights:")
    for i, (name, weight) in enumerate(zip(label_names, weights)):
        print(f"      {i}: {name:6s} = {weight:.4f}")
    
    print(f"\n   ⚠️ AFIB weight: {weights[AFIB_INDEX]:.4f}")
    print(f"   ⚠️ LVH weight: {weights[LVH_INDEX]:.4f}")
    
    if weights[AFIB_INDEX] > 10 or weights[LVH_INDEX] > 10:
        print(f"\n   🔴 WARNING: Class weights quá lớn có thể gây vấn đề!")
else:
    print(f"   No class weights found")

# ===== 5. Test một vài samples thực tế =====
print("\n5️⃣ TESTING SPECIFIC SAMPLES")
print("-" * 50)

# Find indices of AFIB and LVH samples in training set
afib_indices = np.where(y_train[:, AFIB_INDEX] == 1)[0][:5]
lvh_indices = np.where(y_train[:, LVH_INDEX] == 1)[0][:5]

print(f"\n   Testing 5 AFIB samples:")
print(f"   AFIB sample indices: {afib_indices}")

if len(afib_indices) > 0:
    # Load these samples directly
    for idx in afib_indices[:3]:
        try:
            sig = np.load(X_train[idx])
            print(f"      Sample {idx}: shape={sig.shape}, range=[{sig.min():.3f}, {sig.max():.3f}]")
        except Exception as e:
            print(f"      Sample {idx}: ERROR loading - {e}")

print(f"\n   Testing 5 LVH samples:")
print(f"   LVH sample indices: {lvh_indices}")

if len(lvh_indices) > 0:
    for idx in lvh_indices[:3]:
        try:
            sig = np.load(X_train[idx])
            print(f"      Sample {idx}: shape={sig.shape}, range=[{sig.min():.3f}, {sig.max():.3f}]")
        except Exception as e:
            print(f"      Sample {idx}: ERROR loading - {e}")

# ===== 6. Kiểm tra signal range =====
print("\n6️⃣ SIGNAL RANGE CHECK")
print("-" * 50)

sample_signal = signals_batch[0].numpy()
print(f"   First signal shape: {sample_signal.shape}")
print(f"   First signal range: [{sample_signal.min():.4f}, {sample_signal.max():.4f}]")
print(f"   First signal mean: {sample_signal.mean():.4f}")
print(f"   First signal std: {sample_signal.std():.4f}")

if sample_signal.max() > 3.0 or sample_signal.min() < -3.0:
    print(f"\n   ⚠️ WARNING: Signal range bất thường!")
    print(f"      Expected: [-0.5, 0.5] với Constant Scaling")
    print(f"      Got: [{sample_signal.min():.3f}, {sample_signal.max():.3f}]")

print("\n" + "="*80)
print("🔍 DEBUG COMPLETE - Check output above")
print("="*80)

🔍 DEBUGGING - Kiểm tra data và model

1️⃣ LABEL DISTRIBUTION (Training Set)
--------------------------------------------------
   0: SB     = 13203 samples (37.34%)
   1: SR     =  6382 samples (18.05%)
   2: AF     =  6407 samples (18.12%)
   3: AFIB   =  1456 samples ( 4.12%)
   4: SVT    =   546 samples ( 1.54%)
   5: ST     =  5757 samples (16.28%)
   6: LVH    =   528 samples ( 1.49%)

   AFIB index: 3, count: 1456
   LVH index: 6, count: 528

2️⃣ CHECKING FIRST BATCH FROM TRAIN LOADER
--------------------------------------------------
   Batch shape: torch.Size([16, 12, 5000])
   Labels shape: torch.Size([16, 7])

   Classes in this batch:
      0: SB     = 0 samples
      1: SR     = 6 samples
      2: AF     = 3 samples
      3: AFIB   = 1 samples
      4: SVT    = 3 samples
      5: ST     = 2 samples
      6: LVH    = 1 samples

   ✅ AFIB samples in batch: 1
   ✅ LVH samples in batch: 1

3️⃣ CHECKING MODEL OUTPUT
--------------------------------------------------
   Logits sh

---
## 🧠 STEP 8: Load Your Model

### ⚠️ Sử dụng model ResNet18_LSTM_Attn hiện tại của bạn

In [9]:
# ============= LOAD MODEL =============

# 🔧 IMPORT MODEL CỦA BẠN:
# from training.model_resnet18_lstm import ResNet18_LSTM_Attn

# Hoặc define inline nếu cần:
class BasicBlock1D(nn.Module):
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, 
                              padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out


class Attention(nn.Module):
    def __init__(self, input_dim):
        super(Attention, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        weights = self.attn(x)
        weights = F.softmax(weights, dim=1)
        context = torch.sum(weights * x, dim=1)
        return context, weights


class ResNet18_LSTM_Attn(nn.Module):
    def __init__(self, num_classes=7, input_channels=12):
        super(ResNet18_LSTM_Attn, self).__init__()
        self.in_channels = 64
        
        # CNN Backbone
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=7, stride=2, 
                              padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        # LSTM
        self.lstm = nn.LSTM(input_size=512, hidden_size=256, num_layers=2,
                           batch_first=True, bidirectional=True, dropout=0.2)
        
        # Attention
        self.attention = Attention(512)
        
        # Classifier
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, 
                         stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        
        layers = []
        layers.append(BasicBlock1D(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_channels, out_channels))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        # CNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # LSTM
        x = x.permute(0, 2, 1)
        self.lstm.flatten_parameters()
        x, _ = self.lstm(x)
        
        # Attention
        x, weights = self.attention(x)
        
        # Classifier
        out = self.fc(x)
        return out


# Initialize model
num_classes = len(label_names)
model = ResNet18_LSTM_Attn(num_classes=num_classes, input_channels=12)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model loaded: {model.__class__.__name__}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {device}")


✅ Model loaded: ResNet18_LSTM_Attn
   Total parameters: 7,072,136
   Trainable parameters: 7,072,136
   Device: cuda


---
## 🎯 STEP 9: Setup Training with Focal Loss

In [10]:
# ============= CELL 9: TRAINING SETUP (UPDATED) =============

print("\n⚙️ Setting up training with AGGRESSIVE configuration...")

# Step 1: Calculate aggressive weights
print("\n1️⃣ Calculating aggressive class weights...")
class_weights = calculate_aggressive_weights(
    y_train, 
    power=2.0,        # Inverse squared (AGGRESSIVE)
    min_weight=0.1,
    max_weight=50.0
)

print(f"\n   ⭐ Key weights:")
print(f"      AFIB (class {AFIB_INDEX}): {class_weights[AFIB_INDEX]:.2f}")
print(f"      LVH (class {LVH_INDEX}): {class_weights[LVH_INDEX]:.2f}")

# Step 2: Create aggressive focal loss
print("\n2️⃣ Creating AGGRESSIVE Focal Loss...")
criterion = AggressiveFocalLoss(
    alpha=0.25,
    gamma=2.5,  # ← HIGHER than standard 2.0
    class_weights=class_weights
)

print(f"   ✅ AggressiveFocalLoss initialized:")
print(f"      - Alpha: {criterion.alpha}")
print(f"      - Gamma: {criterion.gamma} (standard is 2.0)")
print(f"      - Class weights: Enabled")

# Step 3: Optimizer with LOWER learning rate
print("\n3️⃣ Creating optimizer...")
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005,  # ← REDUCED from 0.001 (stronger gradient from weights)
    weight_decay=1e-5
)

print(f"   ✅ Adam optimizer:")
print(f"      - Learning rate: {optimizer.param_groups[0]['lr']} (was 0.001)")
print(f"      - Weight decay: {optimizer.param_groups[0]['weight_decay']}")

# Step 4: Scheduler
print("\n4️⃣ Creating learning rate scheduler...")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    verbose=True,
    min_lr=1e-6
)

print(f"   ✅ ReduceLROnPlateau:")
print(f"      - Mode: maximize (F1)")
print(f"      - Patience: 5 epochs")
print(f"      - Factor: 0.5")

# Step 5: Verify setup
print("\n5️⃣ Verifying training setup...")

model.train()
test_signals = test_signals[:8].to(device)
test_labels = test_labels[:8].to(device)

try:
    # Forward pass
    optimizer.zero_grad()
    outputs = model(test_signals)
    loss = criterion(outputs, test_labels)
    
    print(f"   Test loss: {loss.item():.4f}")
    
    # Backward pass
    loss.backward()
    
    # Check gradients
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.norm().item() ** 2
    total_norm = total_norm ** 0.5
    
    print(f"   Gradient norm: {total_norm:.4f}")
    
    if total_norm > 0:
        print(f"   ✅ Gradients flowing properly")
    else:
        print(f"   ❌ WARNING: No gradients!")
        
except Exception as e:
    print(f"   ❌ ERROR: {e}")

print("\n" + "="*80)
print("✅ TRAINING SETUP COMPLETE WITH AGGRESSIVE CONFIGURATION")
print("="*80)

print("\n📋 SUMMARY OF CHANGES:")
print("   1. ✅ Oversampling:")
print("      - LVH: 528 → ~5,300 samples (10x)")
print("      - AFIB: 1,456 → ~3,500 samples (2.4x)")
print("   2. ✅ Class weights:")
print(f"      - LVH: 1.91 → {class_weights[LVH_INDEX]:.2f}")
print(f"      - AFIB: 1.15 → {class_weights[AFIB_INDEX]:.2f}")
print("   3. ✅ Focal gamma: 2.0 → 2.5")
print("   4. ✅ Batch size: 32 → 16")
print("   5. ✅ Learning rate: 0.001 → 0.0005")


⚙️ Setting up training with AGGRESSIVE configuration...

1️⃣ Calculating aggressive class weights...

📊 AGGRESSIVE Class Weights (power=2.0)
Class    Samples    Freq %     Weight     vs Avg    
----------------------------------------------------------------------
🟢 0      13203      37.34      0.10       0.1       x
🟢 1      6382       18.05      0.10       0.1       x
🟢 2      6407       18.12      0.10       0.1       x
🟢 3      1456       4.12       0.44       0.4       x
🟡 4      546        1.54       3.13       3.0       x
🟢 5      5757       16.28      0.10       0.1       x
🔴 6      528        1.49       3.35       3.2       x
----------------------------------------------------------------------
Average weight: 1.05
Max weight: 3.35 (class 6)
Min weight: 0.10 (class 0)


   ⭐ Key weights:
      AFIB (class 3): 0.44
      LVH (class 6): 3.35

2️⃣ Creating AGGRESSIVE Focal Loss...
   ✅ AggressiveFocalLoss initialized:
      - Alpha: 0.25
      - Gamma: 2.5 (standard is 2.0)
   

C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


   Test loss: 0.0318
   Gradient norm: 0.0762
   ✅ Gradients flowing properly

✅ TRAINING SETUP COMPLETE WITH AGGRESSIVE CONFIGURATION

📋 SUMMARY OF CHANGES:
   1. ✅ Oversampling:
      - LVH: 528 → ~5,300 samples (10x)
      - AFIB: 1,456 → ~3,500 samples (2.4x)
   2. ✅ Class weights:
      - LVH: 1.91 → 3.35
      - AFIB: 1.15 → 0.44
   3. ✅ Focal gamma: 2.0 → 2.5
   4. ✅ Batch size: 32 → 16
   5. ✅ Learning rate: 0.001 → 0.0005


---
## 🏋️ STEP 10: Training Loop

In [12]:
# ============= TRAINING FUNCTION =============

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    for signals, labels in pbar:
        signals = signals.to(device)
        labels = labels.to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(signals)
        
        loss = criterion(outputs, labels)
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss


def validate(model, val_loader, y_val, device):
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        for signals, _ in tqdm(val_loader, desc="Validating", leave=False):
            signals = signals.to(device)
            outputs = model(signals)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
    
    y_pred_probs = np.vstack(all_probs)
    y_pred_bin = (y_pred_probs >= 0.5).astype(int)
    
    # Calculate metrics
    f1_micro = f1_score(y_val, y_pred_bin, average='micro', zero_division=0)
    f1_macro = f1_score(y_val, y_pred_bin, average='macro', zero_division=0)
    
    # Per-class F1
    f1_per_class = f1_score(y_val, y_pred_bin, average=None, zero_division=0)
    
    return f1_micro, f1_macro, f1_per_class, y_pred_probs


print("✅ Training functions defined")

✅ Training functions defined


In [13]:
# ============= MAIN TRAINING LOOP =============

num_epochs = 50
patience = 10
best_f1 = 0
patience_counter = 0
train_history = []
val_history = []

print(f"\n{'='*80}")
print(f"{'🚀 STARTING TRAINING':^80}")
print(f"{'='*80}")
print(f"   Epochs: {num_epochs}")
print(f"   Patience: {patience}")
print(f"   Device: {device}")
print(f"{'='*80}\n")

for epoch in range(num_epochs):
    print(f"\n📅 Epoch [{epoch+1}/{num_epochs}]")
    
    # Training
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validation
    f1_micro, f1_macro, f1_per_class, val_probs = validate(
        model, val_loader, y_val, device
    )
    
    # Learning rate scheduling
    scheduler.step(f1_micro)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    train_history.append(train_loss)
    val_history.append(f1_micro)
    
    # Print metrics
    print(f"   Loss: {train_loss:.4f} | F1 (micro): {f1_micro:.4f} | F1 (macro): {f1_macro:.4f} | LR: {current_lr:.6f}")
    print(f"   Per-class F1: ", end="")
    for name, f1 in zip(label_names, f1_per_class):
        print(f"{name}={f1:.3f} ", end="")
    print()
    
    # Early stopping and checkpointing
    if f1_micro > best_f1:
        best_f1 = f1_micro
        patience_counter = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_f1': best_f1,
            'f1_per_class': f1_per_class
        }, 'resnetltsm18_050125.pth')
        
        print(f"   ✅ New best model saved! (F1 = {best_f1:.4f})")
    else:
        patience_counter += 1
        print(f"   ⏳ Patience: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"\n⏹️ Early stopping triggered at epoch {epoch+1}")
            break

print(f"\n{'='*80}")
print(f"{'✅ TRAINING COMPLETED':^80}")
print(f"{'='*80}")
print(f"   Best F1 (micro): {best_f1:.4f}")
print(f"   Model saved: best_model_enhanced.pth")
print(f"{'='*80}\n")


                              🚀 STARTING TRAINING                               
   Epochs: 50
   Patience: 10
   Device: cuda


📅 Epoch [1/50]


Training:   0%|          | 0/2606 [00:00<?, ?it/s]

Validating:   0%|          | 0/70 [00:00<?, ?it/s]

   Loss: 0.0026 | F1 (micro): 0.2589 | F1 (macro): 0.1907 | LR: 0.000500
   Per-class F1: SB=0.538 SR=0.000 AF=0.339 AFIB=0.000 SVT=0.273 ST=0.144 LVH=0.042 
   ✅ New best model saved! (F1 = 0.2589)

📅 Epoch [2/50]


Training:   0%|          | 0/2606 [00:00<?, ?it/s]

Validating:   0%|          | 0/70 [00:00<?, ?it/s]

   Loss: 0.0017 | F1 (micro): 0.4133 | F1 (macro): 0.3784 | LR: 0.000500
   Per-class F1: SB=0.906 SR=0.013 AF=0.443 AFIB=0.000 SVT=0.505 ST=0.757 LVH=0.026 
   ✅ New best model saved! (F1 = 0.4133)

📅 Epoch [3/50]


Training:   0%|          | 0/2606 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## 📊 STEP 11: Plot Training History

In [ ]:
# ============= PLOT TRAINING HISTORY =============

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(train_history, label='Train Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# F1 Score
ax2.plot(val_history, label='Val F1 (micro)', marker='o', color='green')
ax2.axhline(y=best_f1, color='r', linestyle='--', label=f'Best F1: {best_f1:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.set_title('Validation F1 Score')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history_enhanced.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved as 'training_history_enhanced.png'")

---
## 🎯 STEP 12: Threshold Tuning on Validation Set

In [ ]:
# ============= THRESHOLD TUNING =============

def find_optimal_thresholds(model, val_loader, y_val, device, 
                           search_range=(0.1, 0.9), num_steps=50):
    """
    Find optimal threshold for each class to maximize F1.
    """
    model.eval()
    
    # Get predictions
    all_probs = []
    with torch.no_grad():
        for signals, _ in tqdm(val_loader, desc="Getting predictions"):
            signals = signals.to(device)
            outputs = model(signals)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
    
    y_pred_probs = np.vstack(all_probs)
    
    # Tune threshold for each class
    n_classes = y_val.shape[1]
    best_thresholds = np.zeros(n_classes)
    
    print(f"\n{'='*80}")
    print(f"{'🔍 THRESHOLD TUNING':^80}")
    print(f"{'='*80}\n")
    
    for i in range(n_classes):
        best_f1 = 0
        best_thresh = 0.5
        
        thresholds = np.linspace(*search_range, num_steps)
        
        for thresh in thresholds:
            y_pred = (y_pred_probs[:, i] >= thresh).astype(int)
            f1 = f1_score(y_val[:, i], y_pred, zero_division=0)
            
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh
        
        best_thresholds[i] = best_thresh
        
        n_pos_pred = (y_pred_probs[:, i] >= best_thresh).sum()
        n_pos_true = y_val[:, i].sum()
        
        print(f"   {label_names[i]:6s}: thresh={best_thresh:.3f}, F1={best_f1:.3f}, "
              f"pred_pos={n_pos_pred:5.0f}, true_pos={n_pos_true:5.0f}")
    
    print(f"\n{'='*80}\n")
    
    return best_thresholds


# Load best model
checkpoint = torch.load('best_model_enhanced.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Best model loaded\n")

# Find optimal thresholds
best_thresholds = find_optimal_thresholds(
    model, val_loader, y_val, device
)

# Save thresholds
np.save('best_thresholds_enhanced.npy', best_thresholds)
print(f"✅ Thresholds saved to 'best_thresholds_enhanced.npy'")

---
## 🧪 STEP 13: Test Set Evaluation (WITH Post-processing)

In [ ]:
# ============= TEST SET EVALUATION =============

print(f"\n{'='*80}")
print(f"{'🧪 TEST SET EVALUATION (WITH POST-PROCESSING)':^80}")
print(f"{'='*80}\n")

model.eval()

# Get predictions and store signals for post-processing
all_probs = []
all_signals = []

with torch.no_grad():
    for signals, _ in tqdm(test_loader, desc="Testing"):
        signals_device = signals.to(device)
        outputs = model(signals_device)
        probs = torch.sigmoid(outputs).cpu().numpy()
        
        all_probs.append(probs)
        all_signals.extend(signals.cpu().numpy())

y_pred_probs = np.vstack(all_probs)

# ===== APPLY POST-PROCESSING FOR AFIB/AF =====
print("\n🔧 Applying AFIB/AF post-processing...")

y_pred_probs_corrected = apply_afib_af_postprocessing(
    y_pred_probs,
    all_signals,
    afib_idx=AFIB_INDEX,
    af_idx=AF_INDEX,
    flutter_threshold=0.3,
    entropy_threshold=2.5,
    sdnn_threshold=100,
    verbose=True
)

# Apply optimal thresholds
y_pred_bin = (y_pred_probs_corrected >= best_thresholds).astype(int)

# ===== METRICS =====
print(f"\n{'='*80}")
print(f"{'📊 CLASSIFICATION REPORT (ENHANCED)':^80}")
print(f"{'='*80}\n")

print(classification_report(
    y_test, y_pred_bin,
    target_names=label_names,
    digits=4,
    zero_division=0
))

# ===== CONFUSION MATRICES =====
print(f"\n{'='*80}")
print(f"{'📈 CONFUSION MATRICES':^80}")
print(f"{'='*80}\n")

cms = multilabel_confusion_matrix(y_test, y_pred_bin)
for i, (cm, name) in enumerate(zip(cms, label_names)):
    print(f"\n🔹 {name}:")
    print(cm)

# ===== COMPARE WITH BASELINE =====
print(f"\n{'='*80}")
print(f"{'📊 COMPARISON WITH BASELINE':^80}")
print(f"{'='*80}\n")

baseline_f1 = {
    'SB': 0.99,
    'SR': 0.92,
    'AF': 0.66,
    'AFIB': 0.33,
    'SVT': 0.81,
    'ST': 0.96,
    'LVH': 0.47
}

_, _, f1_per_class, _ = precision_recall_fscore_support(
    y_test, y_pred_bin, average=None, zero_division=0
)

print(f"{'Class':<10} {'Baseline F1':>12} {'Enhanced F1':>12} {'Improvement':>12}")
print("-" * 50)

for i, name in enumerate(label_names):
    baseline = baseline_f1[name]
    enhanced = f1_per_class[i]
    improvement = enhanced - baseline
    improvement_pct = (improvement / baseline * 100) if baseline > 0 else 0
    
    print(f"{name:<10} {baseline:>12.4f} {enhanced:>12.4f} "
          f"{improvement:>+9.4f} ({improvement_pct:+.1f}%)")

# Calculate overall improvement
baseline_micro = 0.87
enhanced_micro = f1_score(y_test, y_pred_bin, average='micro')
improvement_micro = enhanced_micro - baseline_micro

print("\n" + "-" * 50)
print(f"{'Micro Avg':<10} {baseline_micro:>12.4f} {enhanced_micro:>12.4f} "
      f"{improvement_micro:>+9.4f} ({improvement_micro/baseline_micro*100:+.1f}%)")
print("="* 80 + "\n")

---
## 📈 STEP 14: Visualization

In [ ]:
# ============= VISUALIZE RESULTS =============

# Per-class F1 comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(label_names))
width = 0.35

baseline_f1_list = [baseline_f1[name] for name in label_names]

bars1 = ax.bar(x - width/2, baseline_f1_list, width, label='Baseline', alpha=0.8)
bars2 = ax.bar(x + width/2, f1_per_class, width, label='Enhanced', alpha=0.8)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score Comparison: Baseline vs Enhanced', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(label_names)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('f1_comparison_enhanced.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ F1 comparison plot saved as 'f1_comparison_enhanced.png'")

---
## 💾 STEP 15: Save All Results

In [ ]:
# ============= SAVE RESULTS =============

results = {
    'best_thresholds': best_thresholds,
    'test_predictions': y_pred_bin,
    'test_probabilities': y_pred_probs_corrected,
    'f1_per_class': f1_per_class,
    'f1_micro': enhanced_micro,
    'label_names': label_names,
    'baseline_f1': baseline_f1,
    'improvement': {
        name: f1_per_class[i] - baseline_f1[name]
        for i, name in enumerate(label_names)
    }
}

# Save as numpy archive
np.savez('enhanced_results.npz', **results)

# Save as readable text
with open('enhanced_results_summary.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("ENHANCED TRAINING RESULTS SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("Enhancements Applied:\n")
    f.write("  1. ✅ Constant Scaling (preserves amplitude for LVH)\n")
    f.write("  2. ✅ Class-specific Augmentation (LVH vs Rhythm diseases)\n")
    f.write("  3. ✅ Weighted Focal Loss (handles class imbalance)\n")
    f.write("  4. ✅ Adaptive Balanced Sampler (10x for very rare, 5x for rare)\n")
    f.write("  5. ✅ AFIB/AF Post-processing (feature-based rules)\n")
    f.write("  6. ✅ Per-class Threshold Tuning\n\n")
    
    f.write("=" * 80 + "\n")
    f.write("RESULTS COMPARISON\n")
    f.write("=" * 80 + "\n\n")
    
    f.write(f"{'Class':<10} {'Baseline':>10} {'Enhanced':>10} {'Improvement':>15}\n")
    f.write("-" * 50 + "\n")
    
    for i, name in enumerate(label_names):
        baseline = baseline_f1[name]
        enhanced = f1_per_class[i]
        improvement = enhanced - baseline
        improvement_pct = (improvement / baseline * 100) if baseline > 0 else 0
        
        f.write(f"{name:<10} {baseline:>10.4f} {enhanced:>10.4f} "
               f"{improvement:>+7.4f} ({improvement_pct:+6.1f}%)\n")
    
    f.write("\n" + "-" * 50 + "\n")
    f.write(f"{'Micro Avg':<10} {baseline_micro:>10.4f} {enhanced_micro:>10.4f} "
           f"{improvement_micro:>+7.4f} ({improvement_micro/baseline_micro*100:+6.1f}%)\n")
    
    f.write("\n" + "=" * 80 + "\n")
    f.write("OPTIMAL THRESHOLDS\n")
    f.write("=" * 80 + "\n\n")
    
    for name, thresh in zip(label_names, best_thresholds):
        f.write(f"  {name:<10}: {thresh:.4f}\n")

print("\n✅ All results saved:")
print("   - enhanced_results.npz (binary)")
print("   - enhanced_results_summary.txt (readable)")
print("   - best_model_enhanced.pth (model checkpoint)")
print("   - best_thresholds_enhanced.npy (thresholds)")
print("   - f1_comparison_enhanced.png (visualization)")
print("   - training_history_enhanced.png (training curves)")

print("\n" + "="*80)
print("🎉 TRAINING AND EVALUATION COMPLETED SUCCESSFULLY! 🎉")
print("="*80)